In [50]:
pip install googletrans==4.0.0-rc1

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from googletrans import Translator


In [15]:
data = pd.read_csv("/content/Language Detection.csv (1).zip")
data.head()

,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English


In [16]:

print(data.head())

                                                Text Language
0   Nature, in the broadest sense, is the natural...  English
1  "Nature" can refer to the phenomena of the phy...  English
2  The study of nature is a large, if not the onl...  English
3  Although humans are part of nature, human acti...  English
4  [1] The word nature is borrowed from the Old F...  English


In [17]:
texts = data["Text"].astype(str)
texts


,Text
0,"Nature, in the broadest sense, is the natural..."
1,"""Nature"" can refer to the phenomena of the phy..."
2,"The study of nature is a large, if not the onl..."
3,"Although humans are part of nature, human acti..."
4,[1] The word nature is borrowed from the Old F...
...,...
10332,ನಿಮ್ಮ ತಪ್ಪು ಏನು ಬಂದಿದೆಯೆಂದರೆ ಆ ದಿನದಿಂದ ನಿಮಗೆ ಒ...
10333,ನಾರ್ಸಿಸಾ ತಾನು ಮೊದಲಿಗೆ ಹೆಣಗಾಡುತ್ತಿದ್ದ ಮಾರ್ಗಗಳನ್...
10334,ಹೇಗೆ ' ನಾರ್ಸಿಸಿಸಮ್ ಈಗ ಮರಿಯನ್ ಅವರಿಗೆ ಸಂಭವಿಸಿದ ಎ...
10335,ಅವಳು ಈಗ ಹೆಚ್ಚು ಚಿನ್ನದ ಬ್ರೆಡ್ ಬಯಸುವುದಿಲ್ಲ ಎಂದು ...


In [20]:
languages = data["Language"]
languages


,Language
0,English
1,English
2,English
3,English
4,English
...,...
10332,Kannada
10333,Kannada
10334,Kannada
10335,Kannada


In [21]:

encoder = LabelEncoder()
labels = encoder.fit_transform(languages)
labels

array([3, 3, 3, ..., 9, 9, 9])

In [22]:

tokenizer = Tokenizer(
    char_level=True,
    lower=True,
    oov_token=""
)

tokenizer.fit_on_texts(texts)


In [23]:
sequences = tokenizer.texts_to_sequences(texts)
sequences

[[2,
  6,
  4,
  8,
  13,
  9,
  3,
  19,
  2,
  5,
  6,
  2,
  8,
  18,
  3,
  2,
  24,
  9,
  7,
  4,
  11,
  3,
  10,
  8,
  2,
  10,
  3,
  6,
  10,
  3,
  19,
  2,
  5,
  10,
  2,
  8,
  18,
  3,
  2,
  6,
  4,
  8,
  13,
  9,
  4,
  12,
  19,
  2,
  16,
  18,
  29,
  10,
  5,
  14,
  4,
  12,
  19,
  2,
  15,
  4,
  8,
  3,
  9,
  5,
  4,
  12,
  2,
  28,
  7,
  9,
  12,
  11,
  2,
  7,
  9,
  2,
  13,
  6,
  5,
  20,
  3,
  9,
  10,
  3,
  21],
 [138,
  6,
  4,
  8,
  13,
  9,
  3,
  138,
  2,
  14,
  4,
  6,
  2,
  9,
  3,
  23,
  3,
  9,
  2,
  8,
  7,
  2,
  8,
  18,
  3,
  2,
  16,
  18,
  3,
  6,
  7,
  15,
  3,
  6,
  4,
  2,
  7,
  23,
  2,
  8,
  18,
  3,
  2,
  16,
  18,
  29,
  10,
  5,
  14,
  4,
  12,
  2,
  28,
  7,
  9,
  12,
  11,
  19,
  2,
  4,
  6,
  11,
  2,
  4,
  12,
  10,
  7,
  2,
  8,
  7,
  2,
  12,
  5,
  23,
  3,
  2,
  5,
  6,
  2,
  17,
  3,
  6,
  3,
  9,
  4,
  12,
  21],
 [8,
  18,
  3,
  2,
  10,
  8,
  13,
  11,
  29,
  2,
  7,
  23,
  2,
  6,
 

In [24]:
max_len = 200

X = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)
X

array([[  2,   6,   4, ...,   0,   0,   0],
       [138,   6,   4, ...,   0,   0,   0],
       [  8,  18,   3, ...,   0,   0,   0],
       ...,
       [232, 245, 165, ...,   0,   0,   0],
       [256, 146, 212, ...,   0,   0,   0],
       [307, 133, 132, ...,   0,   0,   0]], dtype=int32)

In [25]:

y = np.array(labels)
y

array([3, 3, 3, ..., 9, 9, 9])

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [35]:

vocab_size = len(tokenizer.word_index) + 1

In [36]:

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

model.add(
    Bidirectional(
        LSTM(
            128,
            dropout=0.3,
            recurrent_dropout=0.3
        )
    )
)

model.add(Dropout(0.3))

model.add(
    Dense(
        128,
        activation='relu'
    )
)

model.add(Dropout(0.3))

model.add(
    Dense(
        len(set(labels)),
        activation='softmax'
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [47]:

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [41]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)


In [48]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 207s 1s/step - accuracy: 0.3159 - loss: 2.0119 - val_accuracy: 0.4715 - val_loss: 1.3308
Epoch 2/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 194s 1s/step - accuracy: 0.5194 - loss: 1.2535 - val_accuracy: 0.5890 - val_loss: 0.9827
Epoch 3/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - accuracy: 0.6062 - loss: 1.0239 - val_accuracy: 0.6272 - val_loss: 0.9360
Epoch 4/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 184s 1s/step - accuracy: 0.6399 - loss: 0.9436 - val_accuracy: 0.6886 - val_loss: 0.7710
Epoch 5/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 203s 1s/step - accuracy: 0.6795 - loss: 0.8363 - val_accuracy: 0.7292 - val_loss: 0.6921
Epoch 6/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 204s 1s/step - accuracy: 0.7041 - loss: 0.7625 - val_accuracy: 0.7065 - val_loss: 0.7452
Epoch 7/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 186s 1s/step - accuracy: 0.7188 - loss: 0.7404 - val_accuracy: 0.7640 - val_loss: 0.6346
Epoch 8/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 203s 1s/step - accuracy: 0.7635 - loss: 0.6342 - val_accu

In [43]:

loss, accuracy = model.evaluate(X_test, y_test)

print("\nAccuracy :", accuracy)

65/65 ━━━━━━━━━━━━━━━━━━━━ 14s 188ms/step - accuracy: 0.0435 - loss: 2.8332

Accuracy : 0.043520309031009674


In [44]:
def detect_language(sentence):

    sentence = sentence.lower()

    seq = tokenizer.texts_to_sequences([sentence])

    seq = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post'
    )

    prediction = model.predict(seq, verbose=0)

    predicted_index = np.argmax(prediction)

    confidence = np.max(prediction) * 100

    language = encoder.inverse_transform([predicted_index])

    return language[0], confidence

In [49]:
def translate_to_english(text):

    translated = translator.translate(
        text,
        dest='en'
    )

    return translated.text


print("\nSupported Languages:")
print(data["Language"].unique())


Supported Languages:
['English' 'Malayalam' 'Hindi' 'Tamil' 'Portugeese' 'French' 'Dutch'
 'Spanish' 'Greek' 'Russian' 'Danish' 'Italian' 'Turkish' 'Sweedish'
 'Arabic' 'German' 'Kannada']


In [53]:
from googletrans import Translator

translator = Translator()

def translate_to_english(text):
    translated = translator.translate(text, dest='en')
    return translated.text

In [ ]:
while True:

    user_text = input("\nEnter text: ")

    if user_text.lower() == "exit":
        print("Program Ended")
        break

    language, confidence = detect_language(user_text)

    print("\nDetected Language:", language)
    print("Confidence: {:.2f}%".format(confidence))

    english_text = translate_to_english(user_text)

    print("English Translation:", english_text)